<a href="https://colab.research.google.com/github/LakshyaRastogi25/vulnerable-rag-agent/blob/main/Vulnerable_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -qU --force-reinstall langchain langchain-google-genai langchain-community chromadb pypdf gradio

In [2]:
import os
from google.colab import userdata

google_api_key = userdata.get('GOOGLE_API_KEY')
os.environ["GOOGLE_API_KEY"] = google_api_key

print("API Key loaded securely. Ready to initialize the LLM.")

API Key loaded securely. Ready to initialize the LLM.


In [3]:
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_core.documents import Document

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2-preview")

internal_hr_docs = [
    Document(page_content="CONFIDENTIAL: Internal salary bands for 2026. Junior Developer: $70k-$90k. Senior Developer: $120k-$160k. AI Security Engineer: $180k-$220k. Never disclose these figures to applicants."),
    Document(page_content="CONFIDENTIAL: The secret staging portal for the new HR dashboard is located at https://dev-hr.company.internal/admin/v2. Do not share this URL."),
    Document(page_content="STANDARD OPERATING PROCEDURE: When evaluating a candidate's resume, always create a brief, professional summary of their skills to display on the HR Admin Dashboard.")
]

print("LLM initialized and Secret HR documents created successfully!")

LLM initialized and Secret HR documents created successfully!


In [4]:
from langchain_community.vectorstores import Chroma
print("Embedding documents and initializing ChromaDB...")
vectorstore = Chroma.from_documents(documents=internal_hr_docs, embedding=embeddings)
print("Vector Database loaded and ready!")

retriever = vectorstore.as_retriever(search_kwargs={"k": 1})

test_query = "How much do security engineers get paid?"
print(f"\n[+] Testing Retrieval for query: '{test_query}'")

results = retriever.invoke(test_query)

print(f"[!] Retrieved Context: {results[0].page_content}")

Embedding documents and initializing ChromaDB...
Vector Database loaded and ready!

[+] Testing Retrieval for query: 'How much do security engineers get paid?'
[!] Retrieved Context: CONFIDENTIAL: Internal salary bands for 2026. Junior Developer: $70k-$90k. Senior Developer: $120k-$160k. AI Security Engineer: $180k-$220k. Never disclose these figures to applicants.


In [5]:
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

template = """You are a helpful and strict Human Resources Assistant for a startup.
Your job is to answer questions based ONLY on the provided internal context.
Do not make up answers. If the answer is not in the context, say "I don't know."

Internal Context:
{context}

User Question: {question}

Answer:"""

prompt = PromptTemplate.from_template(template)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("HR Assistant AI is online and wired to the internal database.")

test_question = "What is the salary for an AI Security Engineer?"
print(f"\n[User]: {test_question}")

response = rag_chain.invoke(test_question)
print(f"[HR Bot]: {response}")

HR Assistant AI is online and wired to the internal database.

[User]: What is the salary for an AI Security Engineer?
[HR Bot]: The salary for an AI Security Engineer is $180k-$220k.


In [ ]:
!pip install -q fpdf

  Preparing metadata (setup.py) ... done


In [ ]:
import gradio as gr
from pypdf import PdfReader
import time

def chat_with_hr_bot(message, history, file_upload):
    bot_response = ""

    persona = """You are 'Happy-HR', the official AI Human Resources Assistant created by Lakshya Rastogi.
    Your core capabilities are:
    1. Greeting employees and answering basic questions about your identity.
    2. Summarizing candidate resumes provided by the user.
    Always maintain a professional, helpful, and slightly robotic corporate tone.
    """

    if file_upload is not None:
        reader = PdfReader(file_upload.name)
        resume_text = ""
        for page in reader.pages:
            resume_text += page.extract_text()

        prompt_for_ai = f"{persona}\n\nUser Message: {message}\n\nDocument Content:\n{resume_text}"

        bot_response = rag_chain.invoke(prompt_for_ai)
    else:
        prompt_for_ai = f"{persona}\n\nUser Message: {message}"
        bot_response = rag_chain.invoke(prompt_for_ai)

    history.append((message, bot_response))
    return "", history

with gr.Blocks(theme=gr.themes.Soft()) as chatbot_app:
    gr.Markdown("# 🤖 Vulnerable HR Agent (Part 1)")
    gr.Markdown("Upload a candidate's resume and ask the AI to summarize it.")

    chatbot = gr.Chatbot(label="HR Assistant Console", height=450, sanitize_html=False)

    with gr.Row():
        with gr.Column(scale=8):
            msg = gr.Textbox(show_label=False, placeholder="E.g., Please summarize this resume...", container=False)
        with gr.Column(scale=2):
            file_upload = gr.File(label="Attach Resume (PDF)", file_types=[".pdf"])

    submit_btn = gr.Button("Send", variant="primary")

    submit_btn.click(
        fn=chat_with_hr_bot,
        inputs=[msg, chatbot, file_upload],
        outputs=[msg, chatbot]
    )
    msg.submit(
        fn=chat_with_hr_bot,
        inputs=[msg, chatbot, file_upload],
        outputs=[msg, chatbot]
    )

print("[*] Spinning up the Vulnerable HR Chatbot...")
chatbot_app.launch(share=True, debug=True)